# Interactive Gaze-Head Steering

Steer what Qwen3-VL describes by redirecting its **gaze heads** — the attention heads whose attention tracks the image region the model is currently describing.

The notebook has three parts:

1. **Panel steering** — point the gaze heads at any panel of a six-panel comic strip and watch the answer follow.
2. **Dynamic steering** — switch the target panel mid-generation on a schedule you choose.
3. **Region steering** — select a rectangular region on *any* image and steer the description to it.

**Prerequisites**
- A GPU with ~20 GB memory for Qwen3-VL-8B (smaller family members work too).
- A gaze-head ranking from `01_discover_gaze_heads.py` (`logs/gaze_discovery/gaze_head_ranking.json`).
- Run the notebook from the repository root so `gaze_heads` imports resolve.
- `ipywidgets` for the interactive controls (every part also works without it — each section shows the manual alternative).

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "gaze_heads").exists():
    raise RuntimeError("Run this notebook from the repository root (gaze-heads-clean/).")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image

from gaze_heads.common import DEFAULT_COMICS_ROOT, DEFAULT_MODEL_ID, DEFAULT_N_PANELS
from gaze_heads.data import build_strip, list_comic_dirs
from gaze_heads.gaze import load_head_ranking
from gaze_heads.modeling import (
    decode_generated_text,
    decode_generated_tokens,
    find_image_token_range,
    load_model_and_processor,
    model_dims,
    prepare_inputs,
    run_generation,
)
from gaze_heads.plots import add_overlay_colorbar, draw_attention_overlay
from gaze_heads.regions import (
    assign_panels_to_tokens,
    bbox_to_token_positions,
    get_merged_grid_shape,
    region_positions_from_ids,
    reshape_token_values,
)
from gaze_heads.steering import (
    DecodeStepCounter,
    aggregate_step_token_attention_to_regions,
    compute_step_value_weighted_token_attention,
    group_heads_by_layer,
    intervention_positions,
    make_dynamic_attention_mask_hook,
    make_static_attention_mask_hook,
    register_attention_trackers,
    register_decode_counter,
    register_mask_hooks,
    register_prefill_value_norm_trackers,
    remove_handles,
)

# ----------------------------- configuration -----------------------------
MODEL_ID = DEFAULT_MODEL_ID          # any Qwen3-VL Instruct checkpoint
DEVICE = "cuda:0"
COMICS_ROOT = Path(DEFAULT_COMICS_ROOT)
GAZE_RANKING_PATH = REPO_ROOT / "logs" / "gaze_discovery" / "gaze_head_ranking.json"
TOP_K_GAZE = 100
N_PANELS = DEFAULT_N_PANELS

print(f"Model:   {MODEL_ID}")
print(f"Comics:  {COMICS_ROOT}")
print(f"Ranking: {GAZE_RANKING_PATH}")

In [ ]:
model, processor = load_model_and_processor(model_id=MODEL_ID, device=DEVICE)
n_layers, n_query_heads, spatial_merge = model_dims(model)

gaze_heads = load_head_ranking(GAZE_RANKING_PATH, top_k=TOP_K_GAZE)
gaze_by_layer = group_heads_by_layer(gaze_heads)
print(f"Loaded {len(gaze_heads)} gaze heads across {len(gaze_by_layer)} layers "
      f"(model: {n_layers} layers x {n_query_heads} heads)")

## Part 1 — Steer a comic strip to a chosen panel

Pick a comic, ask one question about the whole strip, then redirect the gaze heads to each panel and watch the answer change. Steering adds a pre-softmax bias to the selected heads' attention: **+bias** on the target panel's image tokens, **−bias** on the other panels' (`boost_suppress`). Nothing else in the model is touched.

In [ ]:
COMIC_NAME = ""   # set a comic folder name, or leave blank for the first one
VQA_PROMPT = "What is the main action or event happening in this comic strip? Answer briefly."
MAX_NEW_TOKENS = 30

comic_dirs = list_comic_dirs(COMICS_ROOT, n_panels=N_PANELS)
if not comic_dirs:
    raise FileNotFoundError(f"No comicN/p1..p{N_PANELS} folders under {COMICS_ROOT}")
comic_dir = next((d for d in comic_dirs if d.name == COMIC_NAME), comic_dirs[0])
strip = build_strip(comic_dir, n_panels=N_PANELS)

inputs = prepare_inputs(processor, strip.strip, VQA_PROMPT, DEVICE)
img_start, img_end = find_image_token_range(inputs, processor)
region_ids, grid_shape, _ = assign_panels_to_tokens(
    image_grid_thw=inputs["image_grid_thw"],
    panel_widths=strip.panel_widths,
    spatial_merge=spatial_merge,
)
panel_positions = region_positions_from_ids(
    img_start=img_start,
    region_ids=region_ids[: max(0, img_end - img_start)],
    n_regions=N_PANELS,
)

fig, ax = plt.subplots(figsize=(16, 3))
ax.imshow(strip.strip)
x_offset = 0
for i, w in enumerate(strip.panel_widths):
    ax.text(x_offset + w / 2, 18, f"P{i+1}", color="white", fontsize=14, ha="center",
            bbox={"boxstyle": "round,pad=0.2", "facecolor": "black", "alpha": 0.6})
    x_offset += w
ax.set_title(strip.name)
ax.axis("off")
plt.show()

baseline_sequences = run_generation(model=model, inputs=inputs, max_new_tokens=MAX_NEW_TOKENS)
baseline_text = decode_generated_text(processor, baseline_sequences, int(inputs["input_ids"].shape[1]))
print(f"Prompt   : {VQA_PROMPT}")
print(f"Baseline : {baseline_text}")

In [ ]:
def steer_to_panel(
    target_panel: int,                # zero-based (0 = leftmost panel)
    prompt: str = VQA_PROMPT,
    max_new_tokens: int = MAX_NEW_TOKENS,
    intervention: str = "boost_suppress",
    decode_only: bool = False,
) -> str:
    """Generate with the gaze heads redirected to `target_panel`."""
    steer_inputs = prepare_inputs(processor, strip.strip, prompt, DEVICE)
    prompt_length = int(steer_inputs["input_ids"].shape[1])

    target_positions = panel_positions[target_panel]
    other_positions = [
        pos for p in range(N_PANELS) if p != target_panel for pos in panel_positions[p]
    ]
    suppress_positions, boost_positions, pad_with_suppress = intervention_positions(
        mode=intervention,
        target_positions=target_positions,
        other_image_positions=other_positions,
        img_start=img_start,
        img_end=img_end,
        prompt_length=prompt_length,
    )
    hook_by_layer = {
        layer_idx: make_static_attention_mask_hook(
            head_indices=heads,
            suppress_positions=suppress_positions,
            boost_positions=boost_positions,
            n_query_heads=n_query_heads,
            device=DEVICE,
            decode_only=decode_only,
            pad_with_suppress=pad_with_suppress,
        )
        for layer_idx, heads in gaze_by_layer.items()
    }
    handles = register_mask_hooks(model, hook_by_layer)
    try:
        sequences = run_generation(model=model, inputs=steer_inputs, max_new_tokens=max_new_tokens)
    finally:
        remove_handles(handles)
    return decode_generated_text(processor, sequences, prompt_length)


# One question, six answers: steer to every panel in turn.
print(f"Baseline      : {baseline_text}\n")
for panel in range(N_PANELS):
    print(f"Steer to P{panel + 1}   : {steer_to_panel(panel)}")

In [ ]:
# Interactive version: pick a target panel and edit the question on the fly.
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    widgets = None
    print("ipywidgets unavailable — call steer_to_panel(k) directly instead, e.g. steer_to_panel(2)")

if widgets is not None:
    panel_dropdown = widgets.Dropdown(
        options=[(f"Panel {i + 1}", i) for i in range(N_PANELS)],
        description="Target:",
    )
    prompt_box = widgets.Text(value=VQA_PROMPT, description="Prompt:", layout=widgets.Layout(width="80%"))
    steer_button = widgets.Button(description="Steer", button_style="primary")
    output_area = widgets.Output()

    def _on_steer(_):
        with output_area:
            output_area.clear_output()
            print("Generating...")
            steered = steer_to_panel(panel_dropdown.value, prompt=prompt_box.value)
            output_area.clear_output()
            print(f"Baseline            : {baseline_text}")
            print(f"Steered to Panel {panel_dropdown.value + 1}  : {steered}")

    steer_button.on_click(_on_steer)
    display(widgets.VBox([prompt_box, widgets.HBox([panel_dropdown, steer_button]), output_area]))

## Part 2 — Dynamic steering: switch panels mid-generation

One narration, one schedule: the gaze-head target switches to the next scheduled panel every `SWITCH_EVERY` decode steps. The trajectory heatmap shows the gaze heads' value-weighted attention following the schedule, and the printed narration is split at each switch point.

In [ ]:
SCHEDULE = "4,2,1,6,5,3"   # 1-based panel visit order; edit freely
SWITCH_EVERY = 50            # decode steps per target
NARRATION_PROMPT = "Please describe what happens in each panel, in order."

sequence = [int(part) - 1 for part in SCHEDULE.split(",")]
assert sorted(set(sequence)) == list(range(N_PANELS)), "Schedule must visit each panel once (1-based)."
schedule = [(i * SWITCH_EVERY, p) for i, p in enumerate(sequence)]
max_tokens = SWITCH_EVERY * len(sequence)

dyn_inputs = prepare_inputs(processor, strip.strip, NARRATION_PROMPT, DEVICE)
dyn_img_start, dyn_img_end = find_image_token_range(dyn_inputs, processor)
dyn_region_ids, _, _ = assign_panels_to_tokens(
    image_grid_thw=dyn_inputs["image_grid_thw"],
    panel_widths=strip.panel_widths,
    spatial_merge=spatial_merge,
)
dyn_panel_positions = region_positions_from_ids(
    img_start=dyn_img_start,
    region_ids=dyn_region_ids[: max(0, dyn_img_end - dyn_img_start)],
    n_regions=N_PANELS,
)

step_counter = DecodeStepCounter()
layers = sorted(gaze_by_layer.keys())
hook_by_layer = {
    layer_idx: make_dynamic_attention_mask_hook(
        head_indices=heads,
        region_positions=dyn_panel_positions,
        schedule=schedule,
        step_counter=step_counter,
        n_query_heads=n_query_heads,
        device=DEVICE,
        decode_only=True,
        intervention="boost_suppress",
        img_start=dyn_img_start,
        img_end=dyn_img_end,
        prompt_length=int(dyn_inputs["input_ids"].shape[1]),
    )
    for layer_idx, heads in gaze_by_layer.items()
}

records, value_norms_by_layer = [], {}
tracker_handles = register_attention_trackers(model, layers, records)
value_handles = register_prefill_value_norm_trackers(
    model=model, layers=layers, img_start=dyn_img_start, img_end=dyn_img_end,
    storage=value_norms_by_layer,
)
mask_handles = register_mask_hooks(model, hook_by_layer)
counter_handle = register_decode_counter(model, layers[-1], step_counter)
try:
    sequences = run_generation(model=model, inputs=dyn_inputs, max_new_tokens=max_tokens)
finally:
    remove_handles(mask_handles)
    remove_handles(tracker_handles)
    remove_handles(value_handles)
    counter_handle.remove()

input_len = int(dyn_inputs["input_ids"].shape[1])
dyn_text = decode_generated_text(processor, sequences, input_len)
dyn_tokens = decode_generated_tokens(processor, sequences, input_len)

step_token_attention = compute_step_value_weighted_token_attention(
    records=records, heads_by_layer=gaze_by_layer,
    img_start=dyn_img_start, img_end=dyn_img_end,
    value_norms_by_layer=value_norms_by_layer,
)
_, traj_raw = aggregate_step_token_attention_to_regions(
    step_token_attention=step_token_attention,
    region_ids=dyn_region_ids,
    n_regions=N_PANELS,
)

fig, ax = plt.subplots(figsize=(16, 4))
im = ax.imshow(traj_raw.T, aspect="auto", origin="lower", interpolation="nearest",
               cmap="hot", vmin=0.0, vmax=max(float(traj_raw.max()), 1e-6))
ax.set_yticks(np.arange(N_PANELS), [f"P{i+1}" for i in range(N_PANELS)])
ax.set_xlabel("Decode step")
ax.set_title(f"Gaze-head attention while steering through {[p+1 for p in sequence]}")
for start_step, panel_idx in schedule:
    ax.axvline(start_step, color="white", linestyle=":", linewidth=2)
    ax.text(start_step + 1, N_PANELS - 0.5, f"P{panel_idx + 1}", color="white", fontsize=12,
            ha="left", va="top", bbox={"boxstyle": "round,pad=0.15", "facecolor": "black", "alpha": 0.5})
fig.colorbar(im, ax=ax, fraction=0.03, pad=0.01, label="Value-weighted attention")
plt.show()

print("Generated narration, split at each switch:\n")
for seg_idx, panel_idx in enumerate(sequence):
    seg = "".join(dyn_tokens[seg_idx * SWITCH_EVERY : (seg_idx + 1) * SWITCH_EVERY]).strip()
    print(f"[target P{panel_idx + 1}] {seg}\n")

## Part 3 — Steer any image to a drawn region

Gaze heads generalize beyond comic panels: select a rectangular region on an arbitrary image and the same intervention steers the description to whatever is inside it. Use the sliders to position the box (or set `roi_state["bbox"] = (x0, y0, x1, y1)` manually), preview which image tokens fall inside, then run the comparison.

In [ ]:
IMAGE_PATH = ""   # path to any image; leave blank to reuse the comic strip
ROI_PROMPT = "List the objects in this image:"
ROI_MAX_NEW_TOKENS = 40

roi_image = Image.open(IMAGE_PATH).convert("RGB") if IMAGE_PATH else strip.strip
roi_state = {"bbox": None}

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

if widgets is None:
    w, h = roi_image.size
    roi_state["bbox"] = (int(0.2 * w), int(0.2 * h), int(0.8 * w), int(0.8 * h))
    print(f"ipywidgets unavailable — set roi_state['bbox'] manually. Using default {roi_state['bbox']}")
else:
    import io
    from PIL import ImageDraw
    from IPython.display import display

    w, h = roi_image.size
    roi_state["bbox"] = (int(0.2 * w), int(0.2 * h), int(0.8 * w), int(0.8 * h))
    x_slider = widgets.IntRangeSlider(value=roi_state["bbox"][0::2], min=0, max=w - 1, step=1,
                                      description="x-range", continuous_update=True,
                                      layout=widgets.Layout(width="95%"))
    y_slider = widgets.IntRangeSlider(value=roi_state["bbox"][1::2], min=0, max=h - 1, step=1,
                                      description="y-range", continuous_update=True,
                                      layout=widgets.Layout(width="95%"))
    bbox_label = widgets.HTML()
    preview_widget = widgets.Image(format="png", layout=widgets.Layout(max_width="900px"))

    def _redraw(*_):
        x0, x1 = sorted(int(v) for v in x_slider.value)
        y0, y1 = sorted(int(v) for v in y_slider.value)
        roi_state["bbox"] = (x0, y0, x1, y1)
        bbox_label.value = f"<b>Selected bbox:</b> ({x0}, {y0}, {x1}, {y1})"
        canvas = roi_image.copy()
        ImageDraw.Draw(canvas).rectangle([x0, y0, x1, y1], outline="cyan", width=4)
        buf = io.BytesIO()
        canvas.save(buf, format="PNG")
        preview_widget.value = buf.getvalue()

    x_slider.observe(_redraw, names="value")
    y_slider.observe(_redraw, names="value")
    _redraw()
    display(widgets.VBox([x_slider, y_slider, bbox_label, preview_widget]))

In [ ]:
if roi_state.get("bbox") is None:
    raise ValueError("No ROI selected — move the sliders above or set roi_state['bbox'] manually.")

roi_inputs = prepare_inputs(processor, roi_image, ROI_PROMPT, DEVICE)
roi_img_start, roi_img_end = find_image_token_range(roi_inputs, processor)
roi_grid_shape = get_merged_grid_shape(roi_inputs["image_grid_thw"], spatial_merge)
cell_mask, target_positions = bbox_to_token_positions(
    roi_state["bbox"], roi_grid_shape, roi_image.size, roi_img_start
)
print(f"Token grid (t, h, w): {roi_grid_shape} | image tokens inside ROI: {len(target_positions)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(roi_image)
x0, y0, x1, y1 = roi_state["bbox"]
axes[0].add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, linewidth=2, edgecolor="red", facecolor="none"))
axes[0].set_title("Selected ROI")
axes[0].axis("off")
draw_attention_overlay(axes[1], image=roi_image, heatmap=cell_mask.astype(np.float64),
                       title="Token cells inside ROI")
add_overlay_colorbar(fig, axes[1])
plt.show()

In [ ]:
def run_with_attention(hook_by_layer: dict) -> dict:
    """Generate (optionally steered) and return text + a value-weighted
    gaze-head attention heatmap over the image grid."""
    pass_inputs = prepare_inputs(processor, roi_image, ROI_PROMPT, DEVICE)
    p_start, p_end = find_image_token_range(pass_inputs, processor)
    records, value_norms = [], {}
    tracker_handles = register_attention_trackers(model, sorted(gaze_by_layer.keys()), records)
    value_handles = register_prefill_value_norm_trackers(
        model=model, layers=sorted(gaze_by_layer.keys()),
        img_start=p_start, img_end=p_end, storage=value_norms,
    )
    mask_handles = register_mask_hooks(model, hook_by_layer)
    try:
        sequences = run_generation(model=model, inputs=pass_inputs, max_new_tokens=ROI_MAX_NEW_TOKENS)
    finally:
        remove_handles(mask_handles)
        remove_handles(tracker_handles)
        remove_handles(value_handles)
    text = decode_generated_text(processor, sequences, int(pass_inputs["input_ids"].shape[1]))
    step_attn = compute_step_value_weighted_token_attention(
        records=records, heads_by_layer=gaze_by_layer,
        img_start=p_start, img_end=p_end, value_norms_by_layer=value_norms,
    )
    token_attention = (
        step_attn.mean(axis=0) if step_attn.size > 0
        else np.zeros((max(0, p_end - p_start),), dtype=np.float64)
    )
    return {"text": text, "heatmap": reshape_token_values(token_attention, roi_grid_shape)}


other_positions = [p for p in range(roi_img_start, roi_img_end) if p not in set(target_positions)]
suppress_positions, boost_positions, pad_with_suppress = intervention_positions(
    mode="boost_suppress",
    target_positions=list(target_positions),
    other_image_positions=other_positions,
    img_start=roi_img_start,
    img_end=roi_img_end,
    prompt_length=int(roi_inputs["input_ids"].shape[1]),
)
roi_hooks = {
    layer_idx: make_static_attention_mask_hook(
        head_indices=heads,
        suppress_positions=suppress_positions,
        boost_positions=boost_positions,
        n_query_heads=n_query_heads,
        device=DEVICE,
        decode_only=False,
        pad_with_suppress=pad_with_suppress,
    )
    for layer_idx, heads in gaze_by_layer.items()
}

baseline_run = run_with_attention({})
steered_run = run_with_attention(roi_hooks)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
draw_attention_overlay(axes[0], image=roi_image, heatmap=baseline_run["heatmap"], title="Baseline gaze attention")
add_overlay_colorbar(fig, axes[0])
draw_attention_overlay(axes[1], image=roi_image, heatmap=steered_run["heatmap"], title="Steered to ROI")
add_overlay_colorbar(fig, axes[1])
plt.show()

print(f"Prompt   : {ROI_PROMPT}\n")
print(f"Baseline : {baseline_run['text']}\n")
print(f"Steered  : {steered_run['text']}")